# Fashion Product Intelligence V2
## Data Ingestion, Validation and Cleaning

### Objective

- Read product metadata from the raw JSON files
- Load image mapping information
- Merge product metadata with image information
- Validate data quality
- Extract usable category names from nested JSON fields
- Handle missing values and duplicates
- Save the final cleaned dataset

### Input

- Product JSON files: `/Users/jein/Downloads/styles`
- Image mapping file: `/Users/jein/Downloads/images.csv`

### Output

- `fashion_products.csv`
- `cleaned_products.csv`

In [1]:
import os
import json
import pandas as pd
import numpy as np

In [2]:
json_folder = "/Users/jein/env/env/Fashion_product_v2/data_raw/styles"
image_csv_path = "/Users/jein/env/env/Fashion_product_v2/data_raw/images.csv"

In [3]:
# Read and flatten JSON files
product_list = []

files = os.listdir(json_folder)
json_files = [file for file in files if file.endswith(".json")]

print("Total JSON files found:", len(json_files))

for file_name in json_files:
    file_path = os.path.join(json_folder, file_name)

    try:
        with open(file_path, "r") as file:
            json_data = json.load(file)

        product = json_data.get("data", {})

        row = {
            "id": str(product.get("id")),
            "productDisplayName": product.get("productDisplayName"),
            "brandName": product.get("brandName"),
            "gender": product.get("gender"),
            "masterCategory": product.get("masterCategory"),
            "subCategory": product.get("subCategory"),
            "articleType": product.get("articleType"),
            "baseColour": product.get("baseColour"),
            "season": product.get("season"),
            "year": product.get("year"),
            "usage": product.get("usage")
        }

        product_list.append(row)

    except Exception as e:
        print(f"Error reading {file_name}: {e}")

df_products = pd.DataFrame(product_list)

print("Metadata shape:", df_products.shape)

Total JSON files found: 44446
Metadata shape: (44446, 11)


In [4]:
# Load image mapping and merge
df_images = pd.read_csv(image_csv_path)

print("Image mapping shape:", df_images.shape)

df_images["id"] = (
    df_images["filename"]
    .astype(str)
    .str.replace(".jpg", "", regex=False)
)

df_final = pd.merge(
    df_products,
    df_images,
    on="id",
    how="left"
)

df_final.rename(
    columns={"link": "imageURL"},
    inplace=True
)

print("Final dataset shape:", df_final.shape)

Image mapping shape: (44446, 2)
Final dataset shape: (44446, 13)


In [104]:
# Save the merged dataset
df_final.to_csv("fashion_products.csv", index=False)

print("fashion_products.csv saved successfully.")

fashion_products.csv saved successfully.


In [105]:
# initial validation
print("Dataset shape:", df_final.shape)

print("\nColumns:")
print(df_final.columns.tolist())

print("\nFirst 5 records:")
display(df_final.head())

Dataset shape: (44446, 16)

Columns:
['id', 'productDisplayName', 'brandName', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'filename', 'imageURL', 'masterCategoryName', 'subCategoryName', 'articleTypeName']

First 5 records:


,id,productDisplayName,brandName,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,filename,imageURL,masterCategoryName,subCategoryName,articleTypeName
0,57896,Femella Women Off White Shorts,Femella,Women,"{'id': 9, 'typeName': 'Apparel', 'active': Tru...","{'id': 29, 'typeName': 'Bottomwear', 'active':...","{'id': 72, 'typeName': 'Shorts', 'active': Tru...",Off White,Summer,2012,Casual,57896.jpg,http://assets.myntassets.com/v1/images/style/p...,Apparel,Bottomwear,Shorts
1,37297,Casio Enticer Men Black Analogue Watches (A504...,CASIO,Men,"{'id': 1, 'typeName': 'Accessories', 'active':...","{'id': 27, 'typeName': 'Watches', 'active': Tr...","{'id': 66, 'typeName': 'Watches', 'active': Tr...",Black,Winter,2016,Casual,37297.jpg,http://assets.myntassets.com/v1/image/style/pr...,Accessories,Watches,Watches
2,3721,Lee Men Black Canvas Belt,Lee,Men,"{'id': 1, 'typeName': 'Accessories', 'active':...","{'id': 13, 'typeName': 'Belts', 'active': True...","{'id': 49, 'typeName': 'Belts', 'active': True...",Black,Summer,2011,Casual,3721.jpg,http://assets.myntassets.com/v1/images/style/p...,Accessories,Belts,Belts
3,51041,Enamor Women Ivory Coloured Briefs,Enamor,Women,"{'id': 9, 'typeName': 'Apparel', 'active': Tru...","{'id': 136, 'typeName': 'Innerwear', 'active':...","{'id': 137, 'typeName': 'Briefs', 'active': Tr...",White,Summer,2017,Casual,51041.jpg,http://assets.myntassets.com/v1/images/style/p...,Apparel,Innerwear,Briefs
4,2833,Lee Cooper Men's Casual Leather Darknight Blac...,Lee Cooper,Men,"{'id': 10, 'typeName': 'Footwear', 'active': T...","{'id': 33, 'typeName': 'Shoes', 'active': True...","{'id': 93, 'typeName': 'Casual Shoes', 'active...",Black,Summer,2012,Casual,2833.jpg,http://assets.myntassets.com/v1/images/style/p...,Footwear,Shoes,Casual Shoes


In [106]:
# Inspect nested category fields
category_columns = [
    "masterCategory",
    "subCategory",
    "articleType"
]

for column in category_columns:
    value = df_final[column].dropna().iloc[0]

    print(f"\n{column}")
    print("Sample value:", value)
    print("Type:", type(value))


masterCategory
Sample value: {'id': 9, 'typeName': 'Apparel', 'active': True, 'socialSharingEnabled': True, 'isReturnable': True, 'isExchangeable': True, 'pickupEnabled': True, 'isTryAndBuyEnabled': True}
Type: <class 'dict'>

subCategory
Sample value: {'id': 29, 'typeName': 'Bottomwear', 'active': True, 'socialSharingEnabled': True, 'isReturnable': True, 'isExchangeable': True, 'pickupEnabled': True, 'isTryAndBuyEnabled': True}
Type: <class 'dict'>

articleType
Sample value: {'id': 72, 'typeName': 'Shorts', 'active': True, 'socialSharingEnabled': True, 'isReturnable': True, 'isExchangeable': True, 'pickupEnabled': True, 'isTryAndBuyEnabled': True, 'isMyntsEnabled': True, 'serviceabilityDisclaimer': {'title': '', 'desc': ''}}
Type: <class 'dict'>


In [107]:
# Extract category names
def extract_type_name(value):
    if isinstance(value, dict):
        return value.get("typeName")
    return np.nan

df_final["masterCategoryName"] = (
    df_final["masterCategory"]
    .apply(extract_type_name)
)

df_final["subCategoryName"] = (
    df_final["subCategory"]
    .apply(extract_type_name)
)

df_final["articleTypeName"] = (
    df_final["articleType"]
    .apply(extract_type_name)
)

display(
    df_final[
        [
            "masterCategoryName",
            "subCategoryName",
            "articleTypeName"
        ]
    ].head()
)

,masterCategoryName,subCategoryName,articleTypeName
0,Apparel,Bottomwear,Shorts
1,Accessories,Watches,Watches
2,Accessories,Belts,Belts
3,Apparel,Innerwear,Briefs
4,Footwear,Shoes,Casual Shoes


In [108]:
# missing value inspection
missing_summary = pd.DataFrame({
    "Missing Count": df_final.isnull().sum(),
    "Missing Percentage": (
        df_final.isnull().mean() * 100
    ).round(2)
})

display(missing_summary)

,Missing Count,Missing Percentage
id,0,0.0
productDisplayName,0,0.0
brandName,0,0.0
gender,0,0.0
masterCategory,0,0.0
subCategory,0,0.0
articleType,0,0.0
baseColour,0,0.0
season,0,0.0
year,0,0.0


In [109]:
# Duplicate inspection
complex_columns = [
    "masterCategory",
    "subCategory",
    "articleType"
]

analysis_columns = [
    column
    for column in df_final.columns
    if column not in complex_columns
]

duplicate_summary = pd.DataFrame({
    "Check": [
        "Duplicate Product IDs",
        "Duplicate Filenames",
        "Duplicate Analysis Rows"
    ],
    "Count": [
        df_final["id"].duplicated().sum(),
        df_final["filename"].duplicated().sum(),
        df_final[analysis_columns].duplicated().sum()
    ]
})

display(duplicate_summary)

,Check,Count
0,Duplicate Product IDs,0
1,Duplicate Filenames,0
2,Duplicate Analysis Rows,0


In [162]:
text_columns = [
    "productDisplayName",
    "brandName",
    "gender",
    "baseColour",
    "season",
    "usage",
    "filename",
    "imageURL",
    "masterCategoryName",
    "subCategoryName",
    "articleTypeName"
]

for column in text_columns:
    df_final[column] = (
        df_final[column]
        .astype("string")
        .str.strip()
    )

In [ ]:
# Standardize and impute missing text values

missing_tokens = [
    "",
    "NA",
    "N/A",
    "nan",
    "NaN",
    "None",
    "none",
    "NULL",
    "null"
]

for column in text_columns:
    df_final[column] = (
        df_final[column]
        .astype("string")
        .str.strip()
        .replace(missing_tokens, "Unknown")
        .fillna("Unknown")
    )

In [165]:
df_final["year"] = pd.to_numeric(
    df_final["year"],
    errors="coerce"
)

df_final["year"] = df_final["year"].fillna(
    df_final["year"].median()
)

df_final["year"] = df_final["year"].astype(int)

In [166]:
# Handle missing values
fill_unknown_columns = [
    "productDisplayName",
    "baseColour",
    "season",
    "usage"
]

for column in fill_unknown_columns:
    df_final[column] = df_final[column].fillna("Unknown")

In [167]:
# Impute missing values

df_final["productDisplayName"] = df_final["productDisplayName"].replace(
    [pd.NA, None, np.nan, ""],
    "Unknown"
)

df_final["baseColour"] = df_final["baseColour"].replace(
    [pd.NA, None, np.nan, ""],
    "Unknown"
)

df_final["season"] = df_final["season"].replace(
    [pd.NA, None, np.nan, ""],
    "Unknown"
)

df_final["usage"] = df_final["usage"].replace(
    [pd.NA, None, np.nan, ""],
    "Unknown"
)

In [168]:
missing_summary = pd.DataFrame({
    "Missing Count": df_final.isna().sum(),
    "Missing Percentage": (
        df_final.isna().mean() * 100
    ).round(2)
})

display(missing_summary)

,Missing Count,Missing Percentage
id,0,0.0
productDisplayName,0,0.0
brandName,0,0.0
gender,0,0.0
masterCategory,0,0.0
subCategory,0,0.0
articleType,0,0.0
baseColour,0,0.0
season,0,0.0
year,0,0.0


In [ ]:
# # Convert year to numeric
# df_final["year"] = pd.to_numeric(
#     df_final["year"],
#     errors="coerce"
# )

# # Fill missing year with the median year
# df_final["year"] = df_final["year"].fillna(
#     df_final["year"].median()
# )

# # Convert year to integer
# df_final["year"] = df_final["year"].astype(int)

# print("Year data type:", df_final["year"].dtype)
# print("Missing years:", df_final["year"].isna().sum())

Year data type: int64
Missing years: 0


In [169]:
# Remove duplicate products
before = len(df_final)

df_final.drop_duplicates(
    subset="id",
    keep="first",
    inplace=True
)

removed = before - len(df_final)

print("Duplicate products removed:", removed)
print("Current dataset shape:", df_final.shape)

Duplicate products removed: 0
Current dataset shape: (44446, 16)


In [117]:
# final validation
missing_total = df_final.isnull().sum().sum()

duplicate_ids = df_final["id"].duplicated().sum()

duplicate_filenames = df_final["filename"].duplicated().sum()

duplicate_analysis_rows = (
    df_final[analysis_columns]
    .duplicated()
    .sum()
)

print("Final dataset shape:", df_final.shape)
print("Total missing values:", missing_total)
print("Duplicate Product IDs:", duplicate_ids)
print("Duplicate Filenames:", duplicate_filenames)
print(
    "Duplicate analysis rows:",
    duplicate_analysis_rows
)

Final dataset shape: (44446, 16)
Total missing values: 0
Duplicate Product IDs: 0
Duplicate Filenames: 0
Duplicate analysis rows: 0


In [170]:
df_final.to_csv("cleaned_products_v2.csv", index=False)

print("Cleaned dataset saved successfully.")
print("Shape:", df_final.shape)

Cleaned dataset saved successfully.
Shape: (44446, 16)


In [173]:
df_cleaned = pd.read_csv("cleaned_products_v2.csv")

print("Saved dataset shape:", df_cleaned.shape)
print("Missing values:", df_cleaned.isnull().sum().sum())

display(df_cleaned.head())

Saved dataset shape: (44446, 16)
Missing values: 0


,id,productDisplayName,brandName,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,filename,imageURL,masterCategoryName,subCategoryName,articleTypeName
0,57896,Femella Women Off White Shorts,Femella,Women,"{'id': 9, 'typeName': 'Apparel', 'active': Tru...","{'id': 29, 'typeName': 'Bottomwear', 'active':...","{'id': 72, 'typeName': 'Shorts', 'active': Tru...",Off White,Summer,2012,Casual,57896.jpg,http://assets.myntassets.com/v1/images/style/p...,Apparel,Bottomwear,Shorts
1,37297,Casio Enticer Men Black Analogue Watches (A504...,CASIO,Men,"{'id': 1, 'typeName': 'Accessories', 'active':...","{'id': 27, 'typeName': 'Watches', 'active': Tr...","{'id': 66, 'typeName': 'Watches', 'active': Tr...",Black,Winter,2016,Casual,37297.jpg,http://assets.myntassets.com/v1/image/style/pr...,Accessories,Watches,Watches
2,3721,Lee Men Black Canvas Belt,Lee,Men,"{'id': 1, 'typeName': 'Accessories', 'active':...","{'id': 13, 'typeName': 'Belts', 'active': True...","{'id': 49, 'typeName': 'Belts', 'active': True...",Black,Summer,2011,Casual,3721.jpg,http://assets.myntassets.com/v1/images/style/p...,Accessories,Belts,Belts
3,51041,Enamor Women Ivory Coloured Briefs,Enamor,Women,"{'id': 9, 'typeName': 'Apparel', 'active': Tru...","{'id': 136, 'typeName': 'Innerwear', 'active':...","{'id': 137, 'typeName': 'Briefs', 'active': Tr...",White,Summer,2017,Casual,51041.jpg,http://assets.myntassets.com/v1/images/style/p...,Apparel,Innerwear,Briefs
4,2833,Lee Cooper Men's Casual Leather Darknight Blac...,Lee Cooper,Men,"{'id': 10, 'typeName': 'Footwear', 'active': T...","{'id': 33, 'typeName': 'Shoes', 'active': True...","{'id': 93, 'typeName': 'Casual Shoes', 'active...",Black,Summer,2012,Casual,2833.jpg,http://assets.myntassets.com/v1/images/style/p...,Footwear,Shoes,Casual Shoes
